# Spatial econometrics lab\nThis lab is a small, dense-matrix teaching companion. It covers classic and robust OLS-based LM diagnostics, spatial lag 2SLS, spatial Durbin 2SLS, concentrated SAR/SEM/SDM maximum likelihood, and direct/indirect/total impact calculations. Synthetic estimates are teaching demonstrations, not evidence that assumptions hold in applied data.

In [ ]:
import sys
sys.path.insert(0, "implementations/python/spatial-econometrics")
import numpy as np
from diagnostics import spatial_lm_diagnostics
from impacts import spatial_impacts

weights = np.array([
    [0., 1., 0., 0., 0.],
    [1., 0., 1., 0., 0.],
    [0., 1., 0., 1., 0.],
    [0., 0., 1., 0., 1.],
    [0., 0., 0., 1., 0.],
])
x = np.column_stack([np.ones(5), np.arange(5, dtype=float)])
y = np.array([1.0, 2.1, 2.8, 4.2, 4.9])
diagnostics = spatial_lm_diagnostics(y, x, weights)
print("OLS coefficients:", diagnostics.coefficients)
print("LM lag:", diagnostics.lm_lag)
print("LM error:", diagnostics.lm_error)
print("Robust LM lag:", diagnostics.robust_lm_lag)
print("Robust LM error:", diagnostics.robust_lm_error)


## Read the diagnostics carefully

The classic LM-lag and LM-error tests each have an asymptotic chi-square(1) reference under their null. Both can respond to the competing alternative. The Bera-Yoon robust tests adjust for that competing alternative. If the robust results are `None`, the adjustment is not identified for the supplied design and weights. These are diagnostics, not model selection by p-value ranking.

In [ ]:
## Estimate a spatial lag model by 2SLS

The spatially lagged response is endogenous. This synthetic example uses X and WX as instruments. Their validity depends on the data-generating assumptions; the code does not prove instrument exogeneity or relevance.


In [ ]:
from spatial_2sls import spatial_lag_2sls
from maximum_likelihood import spatial_lag_mle, spatial_error_mle
rng = np.random.default_rng(20261115)
n = 80
W_demo = np.zeros((n, n))
for i in range(n - 1):
    W_demo[i, i + 1] = W_demo[i + 1, i] = 1.0
W_demo = W_demo / W_demo.sum(axis=1, keepdims=True)
X_demo = np.column_stack([np.ones(n), rng.normal(size=n)])
y_demo = np.linalg.solve(np.eye(n) - 0.25 * W_demo,
                         X_demo @ np.array([1.0, 0.8]) + rng.normal(scale=0.1, size=n))
iv_fit = spatial_lag_2sls(y_demo, X_demo, W_demo, instrument_orders=1)
sar_fit = spatial_lag_mle(y_demo, X_demo, W_demo)
sem_fit = spatial_error_mle(y_demo, X_demo, W_demo)
print('2SLS rho:', iv_fit.rho)
print('SAR ML rho:', sar_fit.spatial_parameter)
print('SEM ML lambda:', sem_fit.spatial_parameter)


## Estimate a spatial lag model by 2SLS

The spatially lagged response is endogenous. This synthetic example uses X and WX as instruments. Their validity depends on the data-generating assumptions; the code does not prove instrument exogeneity or relevance.

In [ ]:
## Estimate a spatial lag model by 2SLS

The spatially lagged response is endogenous. This synthetic example uses X and WX as instruments. Their validity depends on the data-generating assumptions; the code does not prove instrument exogeneity or relevance.

## Estimate a spatial Durbin model by 2SLS

The SDM includes both `Wy` and `WX`. The spatially lagged response is endogenous; `X` and included `WX` are exogenous regressors. Excluded higher-order lags `W²X`, `W³X`, ... instrument `Wy`. A row-standardised intercept is redundant under spatial lagging and is removed from the included WX columns. Identification and instrument strength remain empirical questions.

In [ ]:
from spatial_durbin_2sls import spatial_durbin_2sls
rng = np.random.default_rng(91827)
n = 250
W = np.zeros((n, n))
for i in range(n - 1):
    W[i, i + 1] = W[i + 1, i] = 1.0
W = W / W.sum(axis=1, keepdims=True)
X = np.column_stack([np.ones(n), rng.normal(size=n)])
rho_true = 0.24
beta_true = np.array([1.1, 0.85])
gamma_true = np.array([0.0, -0.3])
epsilon = rng.normal(scale=0.03, size=n)
y_sdm = np.linalg.solve(np.eye(n) - rho_true * W,
                        X @ beta_true + (W @ X) @ gamma_true + epsilon)
sdm_fit = spatial_durbin_2sls(y_sdm, X, W, instrument_orders=5)
print('estimated rho:', sdm_fit.rho)
print('estimated beta:', sdm_fit.beta)
print('estimated gamma:', sdm_fit.gamma)


## Understand the decomposition

For covariate r, the impact matrix is `S_r = (I - rho W)^-1 (beta_r I + gamma_r W)`. The average direct effect is the mean diagonal; the average total effect is the mean row sum; indirect is total minus direct. The coefficients above are assumed inputs, not estimates from the preceding OLS diagnostic. In a real analysis, report uncertainty for all three impacts using the fitted model's covariance or simulation draws.

## Questions

1. Why is the spatially lagged dependent variable endogenous in the SAR model?
2. Why can classic LM-lag and LM-error tests both respond to the other alternative?
3. What changes in the impact matrix when `rho` is zero?
4. Why can the direct impact differ from `beta` when `rho` is nonzero?
5. Which modelling choices are encoded by the weights matrix, and how would alternative weights change the results?

## Estimate the spatial Durbin model by maximum likelihood\n\nAt each candidate rho, the estimator transforms the response by `I - rho W`, regresses on `[X, WX]`, and adds the log-determinant Jacobian. Spatially lagged columns redundant with X are omitted; with row-standardised weights, the lag of an intercept is exactly redundant.

In [ ]:
from maximum_likelihood import spatial_durbin_mle\nsdm_ml_fit = spatial_durbin_mle(y_sdm, X, W, grid_size=161)\nprint('estimated rho:', sdm_ml_fit.spatial_parameter)\nprint('estimated beta:', sdm_ml_fit.coefficients)\nprint('estimated gamma:', sdm_ml_fit.spatial_covariate_coefficients)\nprint('profile log likelihood:', sdm_ml_fit.log_likelihood)\n

## Estimate a SARAR / SAC model by joint maximum likelihood

The SARAR model contains both response feedback and spatially correlated innovations. For each candidate pair `(rho, lambda)`, the likelihood applies both spatial filters and includes both log-determinants. The search starts from a two-dimensional grid and alternates one-dimensional refinements. The returned structural fitted value is `rho W y + X beta`; the innovation variance is estimated after filtering.


In [ ]:
from maximum_likelihood import spatial_sarar_mle
rng = np.random.default_rng(20261116)
n = 75
W_sarar = np.zeros((n, n))
for i in range(n - 1):
    W_sarar[i, i + 1] = W_sarar[i + 1, i] = 1.0
W_sarar = W_sarar / W_sarar.sum(axis=1, keepdims=True)
X_sarar = np.column_stack([np.ones(n), rng.normal(size=n)])
rho_true, lambda_true = 0.22, 0.38
innovation = rng.normal(scale=0.12, size=n)
error = np.linalg.solve(np.eye(n) - lambda_true * W_sarar, innovation)
y_sarar = np.linalg.solve(np.eye(n) - rho_true * W_sarar,
                          X_sarar @ np.array([1.2, -0.55]) + error)
sarar_fit = spatial_sarar_mle(y_sarar, X_sarar, W_sarar, grid_size=31)
print('estimated rho:', sarar_fit.spatial_parameter)
print('estimated lambda:', sarar_fit.error_parameter)
print('estimated beta:', sarar_fit.coefficients)
print('innovation variance:', sarar_fit.sigma2_mle)


## Quantify uncertainty in spatial impacts

Supply the joint covariance of `[rho, beta..., gamma...]` from a fitted estimator or another justified source. The routine simulates parameter vectors, recalculates all impact matrices, and reports percentile intervals and Monte Carlo standard deviations. It rejects ill-conditioned spatial systems; it does not estimate the covariance matrix or guarantee that a normal approximation is appropriate.


In [ ]:
from impacts import spatial_impact_uncertainty
# Illustrative covariance only; in applied work use the fitted model's joint covariance.
parameter_covariance = np.diag([0.01, 0.04, 0.04, 0.04, 0.04])
uncertainty = spatial_impact_uncertainty(
    W, rho, beta, parameter_covariance, gamma,
    draws=2000, seed=2026,
)
for index, item in enumerate(uncertainty):
    print(index, 'direct:', item.direct)
    print(index, 'indirect:', item.indirect)
    print(index, 'total:', item.total)
